In [31]:
import pandas as pd

In [32]:
import re

data = pd.read_csv("twcs.csv", dtype={
    "tweet_id": "string",
    "author_id": "string",
    "response_tweet_id": "string",
    "in_response_to_tweet_id": "string"
})

In [33]:
text_values = data["text"].fillna("")

leading_mentions = text_values.str.extract(
    r"^\s*((?:@[A-Za-z0-9_]+\s*)+)",
    expand=False
).fillna("")

data["receiver_id"] = leading_mentions.apply(
    lambda value: ", ".join(re.findall(r"@[A-Za-z0-9_]+", value))
)

data["clean_text"] = text_values.str.replace(
    r"^\s*(?:@[A-Za-z0-9_]+\s*)+",
    "",
    regex=True
).str.strip()

In [34]:
brand = "XboxSupport"

xbox_data = data[
    data["author_id"].eq(brand)
    | data["receiver_id"].str.contains(
        rf"(?<!\w)@?{brand}(?!\w)",
        regex=True,
        na=False
    )
].copy()

In [35]:
xbox_customer_messages = xbox_data[
    xbox_data["inbound"].eq(True)
].copy()

xbox_replies = xbox_data[
    xbox_data["author_id"].eq(brand)
].copy()

In [36]:
reply_lookup = xbox_replies.set_index("tweet_id")[
    ["clean_text", "created_at"]
].rename(columns={
    "clean_text": "historical_reply",
    "created_at": "reply_created_at"
})

training_data = xbox_customer_messages.join(
    reply_lookup,
    on="response_tweet_id"
)

training_data = training_data.dropna(
    subset=["clean_text", "historical_reply"]
)

training_data[
    ["tweet_id", "clean_text", "historical_reply"]
].head()

,tweet_id,clean_text,historical_reply
189,277,can I change me sons Xbox live account to his ...,"Hi, you can change your Microsoft account emai..."
220,310,They have no info either..,We'd recommend keeping an eye out on your emai...
227,311,Would like to know so I can make sure I have e...,Hi there! We'd recommend reaching out to the c...
230,320,redeemed a code for fifa points this afternoon...,Hello! Would you mind following us and sending...
233,323,the 5 app on Xboxes not working error code 200...,Can you show us what is appearing on your scre...


In [37]:
training_data = training_data.drop_duplicates(
    subset=["clean_text", "historical_reply"]
)

training_data = training_data[
    training_data["clean_text"].str.len().between(5, 1000)
]

In [40]:
training_data = training_data.drop(
    columns=["text", "created_at", "reply_created_at"],
    errors="ignore"
)

training_data.head()

,tweet_id,author_id,inbound,response_tweet_id,in_response_to_tweet_id,receiver_id,clean_text,historical_reply
189,277,115771,True,276,<NA>,@XboxSupport,can I change me sons Xbox live account to his ...,"Hi, you can change your Microsoft account emai..."
220,310,115785,True,312,309,@XboxSupport,They have no info either..,We'd recommend keeping an eye out on your emai...
227,311,115785,True,309,318,"@XboxSupport, @115786, @115787, @15913",Would like to know so I can make sure I have e...,Hi there! We'd recommend reaching out to the c...
230,320,115789,True,319,<NA>,"@115790, @XboxSupport",redeemed a code for fifa points this afternoon...,Hello! Would you mind following us and sending...
233,323,115791,True,321,<NA>,"@1520, @XboxSupport",the 5 app on Xboxes not working error code 200...,Can you show us what is appearing on your scre...


In [41]:
# Remove missing and duplicate training pairs
training_data = training_data.dropna(
    subset=["clean_text", "historical_reply"]
)

training_data = training_data.drop_duplicates(
    subset=["clean_text", "historical_reply"]
)

# Normalize whitespace
training_data["clean_text"] = (
    training_data["clean_text"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

training_data["historical_reply"] = (
    training_data["historical_reply"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Remove empty values after cleaning
training_data = training_data[
    training_data["clean_text"].ne("")
    & training_data["historical_reply"].ne("")
].copy()

training_data.shape

(13469, 8)

In [42]:
training_data.columns.tolist()
training_data[["clean_text", "historical_reply"]].head()

,clean_text,historical_reply
189,can I change me sons Xbox live account to his ...,"Hi, you can change your Microsoft account emai..."
220,They have no info either..,We'd recommend keeping an eye out on your emai...
227,Would like to know so I can make sure I have e...,Hi there! We'd recommend reaching out to the c...
230,redeemed a code for fifa points this afternoon...,Hello! Would you mind following us and sending...
233,the 5 app on Xboxes not working error code 200...,Can you show us what is appearing on your scre...


In [43]:
training_data.to_csv(
    "xbox_training_data.csv",
    index=False,
    encoding="utf-8"
)